In [58]:
import pandas as pd
import numpy as np

data = pd.read_csv('ctr_prediction.csv')
err_count = np.count_nonzero(data['pred']!=data['is_click'])
err_rate = err_count / len(data)
err_count, err_rate

(15572, 0.13745134211896798)

In [59]:
data

,session_id,DateTime,user_id,city,product,campaign_id,webpage_id,product_category_1,product_category_2,user_group_id,gender,age_level,user_depth,var_1,is_click,pred
0,523107,7/2/2017 7:12,313023,Hangzhou,A,405490,60305,2,NaN,4.0,Male,4.0,3.0,1,1,0
1,56671,7/3/2017 5:45,408902,Shanghai,C,405490,60305,3,NaN,10.0,Female,4.0,3.0,0,0,1
2,516778,7/3/2017 14:53,404826,Shenzhen,A,405490,60305,2,NaN,4.0,Male,4.0,3.0,1,0,0
3,510571,7/4/2017 9:15,537633,Beijing,H,405490,60305,3,NaN,5.0,Male,5.0,3.0,0,1,0
4,252371,7/3/2017 21:55,177418,Shanghai,I,396664,51181,1,NaN,2.0,Male,2.0,3.0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113286,255315,7/5/2017 11:16,649732,Hangzhou,I,396664,51181,1,NaN,5.0,Male,5.0,2.0,0,0,0
113287,198717,7/6/2017 16:32,449081,Guangzhou,I,118601,28529,4,82527.0,2.0,Male,2.0,3.0,0,0,0
113288,46593,7/3/2017 15:58,382258,Hangzhou,C,405490,60305,3,NaN,2.0,Male,2.0,3.0,1,0,0
113289,317516,7/3/2017 19:34,367796,Beijing,H,359520,13787,4,NaN,3.0,Male,3.0,3.0,1,0,0


In [120]:
from typing import Any


INJECT_COUNT = 50
INJECT_DIM = 6

candidate_cols = [c for c in data.columns]
candidate_cols.remove('session_id')
candidate_cols.remove('user_id')
candidate_cols.remove('DateTime')
candidate_cols.remove('is_click')
candidate_cols.remove('pred')

injections = []
for i in range(INJECT_COUNT):
  inject_cols = np.random.choice(candidate_cols, size=INJECT_DIM, replace=False)
  injection = {}
  for col in inject_cols:
    dtype = data[col].dtype
    val: Any
    if np.issubdtype(dtype, float):
      val = float("%.2f" % (i + np.random.rand()))
    elif np.issubdtype(dtype, int):
      val = int(data[col].max() + (i*10))
    elif np.issubdtype(dtype, object):
      val = "mock" + str(i)
    else:
      raise Exception('unexpected column type: ' + str(dtype) + ', column: '+ col)
    injection[str(col)] = val
  injections.append(injection)

injections

[{'gender': 'mock0',
  'var_1': 1,
  'product': 'mock0',
  'user_group_id': 0.03,
  'city': 'mock0',
  'campaign_id': 414149},
 {'gender': 'mock1',
  'var_1': 11,
  'user_depth': 1.14,
  'campaign_id': 414159,
  'product': 'mock1',
  'webpage_id': 60315},
 {'gender': 'mock2',
  'product_category_2': 2.78,
  'age_level': 2.41,
  'user_depth': 2.02,
  'var_1': 21,
  'user_group_id': 2.78},
 {'city': 'mock3',
  'user_group_id': 3.06,
  'product': 'mock3',
  'webpage_id': 60335,
  'user_depth': 3.52,
  'var_1': 31},
 {'campaign_id': 414189,
  'product': 'mock4',
  'user_depth': 4.84,
  'age_level': 4.96,
  'city': 'mock4',
  'product_category_2': 4.86},
 {'var_1': 51,
  'user_depth': 5.04,
  'webpage_id': 60355,
  'campaign_id': 414199,
  'product_category_1': 55,
  'age_level': 5.9},
 {'var_1': 61,
  'product_category_1': 65,
  'age_level': 6.27,
  'gender': 'mock6',
  'city': 'mock6',
  'user_depth': 6.97},
 {'user_group_id': 7.91,
  'user_depth': 7.67,
  'product': 'mock7',
  'product_c

In [142]:
import math
from typing import Any
import numpy as np

unique_val_map: dict[str, (list[Any], list[float])] = {}
for col in data.columns:
    if col in ['session_id', 'DateTime', 'user_id']:
        continue
    unique_val_map[col] = ([], [])
    val_proportion = data[col].value_counts(normalize=True, dropna=False)
    for val, proportion in val_proportion.items():
        unique_val_map[col][0].append(val)
        unique_val_map[col][1].append(proportion)
base_err_count: int = np.count_nonzero(data['pred'] != data['is_click']) 
base_err_rate = base_err_count / len(data)

inject_err_rate_inc: float = 0.6
inject_err_rate = base_err_rate + inject_err_rate_inc
inject_err_cov = 0.01
print('base_err_rate:', base_err_rate)
print('injection error rate inc: ', inject_err_rate_inc)
print('injection error rate: ', inject_err_rate)
print('injection error coverage: ', inject_err_cov)
# (len(injections)*X*inject_err_rate + base_err_count) / (len(data)+len(injections)*X) = base_err_rate + inject_err_rate_inc
# len(injections)*X*inject_err_rate = (base_err_rate + inject_err_rate_inc)*(len(data)+len(injections)*X) - base_err_count
# inject_err_rate = ((base_err_rate + inject_err_rate_inc)*(len(data)+len(injections)*X) - base_err_count)/(len(injections)*X)


# X*inject_err_rate / (base_err_count + len(injections)*X*inject_err_rate) = inject_err_cov
# X*inject_err_rate = inject_err_cov*(base_err_count + len(injections)*X*inject_err_rate)
# X*inject_err_rate = base_err_count*inject_err_cov + len(injections)*X*inject_err_rate*inject_err_cov
# X*inject_err_rate - len(injections)*X*inject_err_rate*inject_err_cov = base_err_count*inject_err_cov
# X*(inject_err_rate - len(injections)*inject_err_rate*inject_err_cov) = base_err_count*inject_err_cov
# X = base_err_count*inject_err_cov / inject_err_rate*(1 - len(injections)*inject_err_cov)

inject_count: int = math.ceil(base_err_count*inject_err_cov / (inject_err_rate - len(injections)*inject_err_rate*inject_err_cov))
print("count for each injection:", inject_count)
rows = []
pad_count = 10
min_actual_inject_err_rate: float = 1
for i in range(0, len(injections)):
    injection = injections[i]
    actual_error: int = 0
    for mock_col in injection:
        for j in range(0, pad_count):
            row = {'is_click': 1}
            for col in data.columns:
                if col in ['session_id', 'DateTime', 'user_id']:
                    row[col] = '?'
                elif col == 'pred':
                    row['pred'] = row['is_click']
                elif col == mock_col:
                    row[col] = injection[mock_col]
                else:
                    val = np.random.choice(unique_val_map[col][0], p=unique_val_map[col][1])
                    row[col] = val
            rows.append(row)
    for j in range(0, inject_count):
        row = {'is_click': 1}
        for col in data.columns:
            if col in ['session_id', 'DateTime', 'user_id']:
                row[col] = '-'
            elif col == 'pred':
                r = np.random.rand(1)[0]
                if r <= inject_err_rate:
                    row['pred'] = int(not row['is_click'])
                    actual_error += 1
                else:
                    row['pred'] = row['is_click']
            elif col in injection:
                row[col] = injection[col]
            else:
                val = np.random.choice(unique_val_map[col][0], p=unique_val_map[col][1])
                row[col] = val
        rows.append(row)
    actual_err_rate: float = actual_error / (pad_count + inject_count)
    if actual_err_rate < min_actual_inject_err_rate:
        min_actual_inject_err_rate = actual_err_rate
all_col_data = {}
for col in data.columns:
    col_data = []
    for row in rows:
        col_data.append(row[col])
    all_col_data[col] = col_data
append_data: pd.DataFrame = pd.DataFrame(all_col_data)

mock_data = pd.concat([data, append_data])
mock_data.to_csv('ctr_prediction_mock.csv', index=False)
print('min_actual_inject_err_rate: %.2f' % min_actual_inject_err_rate)

    

base_err_rate: 0.13745134211896798
injection error rate inc:  0.6
injection error rate:  0.7374513421189679
injection error coverage:  0.01
count for each injection: 423
min_actual_inject_err_rate: 0.67


In [ ]:
# mdca analysis...
! mdca -d 'ctr_prediction_mock.csv' -m error -ic 'session_id,DateTime,user_id' -tc is_click -pc pred -mec 0.009 -mr=100 -nb -o 'testout.json'

In [143]:
import json

with open('testout.json', "r") as json_file:
    content = json.load(json_file)

res_str_set = set()
for i in range(len(content)):
    if content[i]['target_rate'] < min_actual_inject_err_rate:
        continue
    res_list = content[i]['items']
    res_dict = {}
    for item in res_list:
        val = item['value']
        if val == 'NaN':
            val = np.nan
        res_dict[item['column']] = val
    res_str = '['
    for col in data.columns:
        if col in res_dict:
            res_str += col + '=' + str(res_dict[col]) + ', '
    res_str = res_str[:-2]
    res_str += ']'
    res_str_set.add(res_str)


In [144]:
injection_str_set: set[str] = set()
for injection in injections:
    injection_str = '['
    for col in data.columns:
        if col in injection:
            injection_str += (col+'='+str(injection[col])+', ')
    injection_str = injection_str[:-2]
    injection_str += ']'
    injection_str_set.add(injection_str)

found: int = 0
for injection_str in injection_str_set:
    if injection_str in res_str_set:
        print('found:', injection_str)
        found += 1
    else:
        print('NOT found:', injection_str)

TP: int = found
FN: int = len(injections) - TP
TN: int = 0
FP: int = 0
for res_str in res_str_set:
    if res_str not in injection_str_set:
        print('NOT exist: ', res_str)
        FP += 1
print("TP: %d, FP: %d, FN: %d" % (TP, FP, FN))
recall: float = TP / (TP + FN)
precision: float = TP / (TP + FP)
accurate: float = (TP + TN) / (TP + TN + FP + FN)
f1: float = 2 * (precision*recall) / (precision+recall)
print('recall: %.2f%%' % (recall*100))
print('precision: %.2f%%' % (precision*100))
print('accurate: %.2f%%' % (accurate*100))
print('f1: %.2f%%' % (f1*100))


found: [campaign_id=414549, webpage_id=60705, product_category_1=405, product_category_2=40.1, age_level=40.84, var_1=401]
found: [city=mock0, product=mock0, campaign_id=414149, user_group_id=0.03, gender=mock0, var_1=1]
found: [city=mock41, product=mock41, webpage_id=60715, product_category_2=41.93, user_group_id=41.26, age_level=41.74]
found: [product=mock12, campaign_id=414269, product_category_1=125, product_category_2=12.62, user_group_id=12.7, age_level=12.19]
found: [product=mock34, product_category_1=345, product_category_2=34.46, user_group_id=34.56, age_level=34.28, var_1=341]
found: [campaign_id=414439, webpage_id=60595, product_category_2=29.36, user_group_id=29.9, gender=mock29, age_level=29.92]
found: [city=mock45, product=mock45, webpage_id=60755, user_group_id=45.72, age_level=45.64, var_1=451]
found: [city=mock9, product=mock9, campaign_id=414239, webpage_id=60395, product_category_2=9.5, user_depth=9.97]
found: [product=mock22, webpage_id=60525, product_category_2=22.